In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv("Churn_Modelling.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [4]:
df.head(50)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,10,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [5]:
df[['Geography','Gender',]].nunique()

Geography    3
Gender       2
dtype: int64

In [6]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [7]:
df.drop(columns = ['RowNumber','CustomerId','Surname'],inplace = True)

In [8]:
X = df.drop(columns = 'Exited')
y = df['Exited']

In [9]:
X_train,X_test,y_train,y_test = train_test_split(
    X , y , random_state = 42 , test_size = 0.2
)

In [10]:
cat_cols = ['Gender','Geography']

In [11]:
cat_pip = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown = 'ignore')),
    ('scaler', StandardScaler(with_mean = False))
])

In [12]:
data = ColumnTransformer([
    ("cat", cat_pip, cat_cols)
])

In [13]:
X_train_ready = data.fit_transform(X_train)
X_test_ready = data.transform(X_test)

In [14]:
X_train_ready.shape

(8000, 5)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [16]:
model = Sequential([
    Input(shape = (X_train_ready.shape[1],)),

    Dense(64,activation='relu'),
    Dense(32,activation='relu'),

    Dense(1,activation='sigmoid')
])

In [17]:
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

In [18]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [19]:
history = model.fit(
    X_train_ready, y_train,        
    epochs=200,                    
    batch_size=64,                 
    validation_split=0.2,          
    callbacks=[early_stop],        
    verbose=1                      
)

Epoch 1/200
100/100 [==============================] - 0s 846us/step - loss: 0.5103 - accuracy: 0.7794 - val_loss: 0.4850 - val_accuracy: 0.7987
Epoch 2/200
100/100 [==============================] - 0s 409us/step - loss: 0.4922 - accuracy: 0.7934 - val_loss: 0.4841 - val_accuracy: 0.7987
Epoch 3/200
100/100 [==============================] - 0s 390us/step - loss: 0.4923 - accuracy: 0.7934 - val_loss: 0.4835 - val_accuracy: 0.7987
Epoch 4/200
100/100 [==============================] - 0s 414us/step - loss: 0.4922 - accuracy: 0.7934 - val_loss: 0.4818 - val_accuracy: 0.7987
Epoch 5/200
100/100 [==============================] - 0s 395us/step - loss: 0.4927 - accuracy: 0.7934 - val_loss: 0.4827 - val_accuracy: 0.7987
Epoch 6/200
100/100 [==============================] - 0s 444us/step - loss: 0.4922 - accuracy: 0.7934 - val_loss: 0.4834 - val_accuracy: 0.7987
Epoch 7/200
100/100 [==============================] - 0s 394us/step - loss: 0.4916 - accuracy: 0.7934 - val_loss: 0.4839 - val_ac

In [20]:
loss, accuracy = model.evaluate(X_test_ready, y_test, verbose=0) 
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 80.35%
